In [1]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import json
import pandas as pd

In [2]:
path_dataset_train = "../../../../2025 EXIST/EXIST 2025 Tweets Dataset/training/EXIST2025_training.json"

In [3]:
def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
        
    df_raw = pd.DataFrame.from_dict(raw_data, orient="index").reset_index()
    df_raw = df_raw.rename(columns={"index": "id"})
    return df_raw

In [4]:
data_train = load_data(path_dataset_train)

In [5]:
path_hard_train1_1 = "../../../../2025 EXIST/evaluation/golds/EXIST2025_training_task1_1_gold_hard.json"

In [6]:
def load_eval(path):
    with open(path, 'r', encoding='utf-8') as f:
        data_from_file = json.load(f)

    df = pd.DataFrame(data_from_file)
    return df

In [7]:
classes_task1_1 = ['YES', 'NO']

In [8]:
# Load soft and hard evaluations from disk
hard_train1_1 = load_eval(path_hard_train1_1)

# Rename target columns
hard_train1_1 = hard_train1_1.rename(columns={'value': 'hard_train1_1'})

# One-hot encode based on the string values in the column
hard_train1_1_one_hot_df = pd.get_dummies(hard_train1_1['hard_train1_1']).reindex(columns=classes_task1_1, fill_value=0)
hard_train1_1 = pd.concat([hard_train1_1, hard_train1_1_one_hot_df], axis=1)

# Append encoded evaluations to dataset
data_train = pd.merge(data_train, hard_train1_1[['id','YES', 'NO']], on='id', how='left')

In [9]:
data_train_en = data_train[data_train['lang']=='en']

In [10]:
data_train_small = data_train_en[['tweet', 'YES', 'NO']]

In [11]:
data_train_small.describe()

,tweet,YES,NO
count,3260,2870,2870
unique,3260,2,2
top,FFS! How about laying the blame on the bastard...,False,True
freq,1,1733,1733


In [12]:
df_train = data_train_small.copy()
df_train = df_train[(df_train['YES'].notna()) | (df_train['NO'].notna())] 
df_train = df_train.rename(columns={'tweet': 'text'})

In [13]:
df_train[:10]

,text,YES,NO
3661,Writing a uni essay in my local pub with a cof...,True,False
3662,@UniversalORL it is 2021 not 1921. I dont appr...,True,False
3665,According to a customer I have plenty of time ...,True,False
3666,"So only 'blokes' drink beer? Sorry, but if you...",True,False
3667,New to the shelves this week - looking forward...,False,True
3669,I guess that’s fairly normal for a Neanderthal...,False,True
3670,#EverydaySexism means women usually end up in ...,True,False
3672,@orlamuldoon @NWCI @IrishRunnerMag @ReclaimTS ...,True,False
3674,@MarkPaulTimes @colettebrowne #EveryDaySexism ...,True,False
3675,@RMatthewsPsyEdu @ITV @jamesmartinchef @Everyd...,True,False


In [14]:
path_dataset_eval = "../../../../2025 EXIST/EXIST 2025 Tweets Dataset/dev/EXIST2025_dev.json"

In [15]:
data_eval = load_data(path_dataset_eval)

In [16]:
path_hard_eval1_1 = "../../../../2025 EXIST/evaluation/golds/EXIST2025_dev_task1_1_gold_hard.json"

In [17]:
# Load soft and hard evaluations from disk
hard_eval1_1 = load_eval(path_hard_eval1_1)

# Rename target columns
hard_eval1_1 = hard_eval1_1.rename(columns={'value': 'hard_eval1_1'})

# One-hot encode based on the string values in the column
hard_eval1_1_one_hot_df = pd.get_dummies(hard_eval1_1['hard_eval1_1']).reindex(columns=classes_task1_1, fill_value=0)
hard_eval1_1 = pd.concat([hard_eval1_1, hard_eval1_1_one_hot_df], axis=1)

# Append encoded evaluations to dataset
data_eval = pd.merge(data_eval, hard_eval1_1[['id','YES', 'NO']], on='id', how='left')

In [18]:
data_eval_en = data_eval[data_eval['lang']=='en']

In [19]:
data_eval_small = data_eval_en[['tweet', 'YES', 'NO']]

In [20]:
data_eval_small.describe()

,tweet,YES,NO
count,489,444,444
unique,489,2,2
top,"@Mike_Fabricant “You should smile more, love. ...",False,True
freq,1,250,250


In [21]:
df_eval = data_eval_small.copy()
df_eval = df_eval[(df_eval['YES'].notna()) | (df_eval['NO'].notna())] 
df_eval = df_eval.rename(columns={'tweet': 'text'})

In [22]:
df_eval[:10]

,text,YES,NO
549,"@Mike_Fabricant “You should smile more, love. ...",False,True
550,@BBCWomansHour @LabWomenDec @EverydaySexism Sh...,True,False
551,#everydaysexism Some man moving my suitcase in...,True,False
552,@KolHue @OliverJia1014 lol gamergate the go to...,False,True
553,@ShelfStoriesGBL To me this has the same negat...,False,True
554,@IrrelevantCmnt @jbo911 @BanButterfly @TheRigh...,False,True
555,@cathymwafer @andrew_lilico Showing off? The m...,False,True
556,@ReproRights @AbortionStories Getting Twitter ...,True,False
557,@shields_rex @good_jarvis4 @Lulu48005877 @Mart...,False,True
559,@esjayXX @EcuadorianMum @monsalore They so rem...,True,False


In [23]:
len(df_eval)

444

In [24]:
df_train.describe()

,text,YES,NO
count,2870,2870,2870
unique,2870,2,2
top,Writing a uni essay in my local pub with a cof...,False,True
freq,1,1733,1733


In [25]:
import gensim.downloader as api
word2vec = api.load("glove-twitter-100")

In [26]:
df_eval[:5]

,text,YES,NO
549,"@Mike_Fabricant “You should smile more, love. ...",False,True
550,@BBCWomansHour @LabWomenDec @EverydaySexism Sh...,True,False
551,#everydaysexism Some man moving my suitcase in...,True,False
552,@KolHue @OliverJia1014 lol gamergate the go to...,False,True
553,@ShelfStoriesGBL To me this has the same negat...,False,True


In [59]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import sys
import pandas as pd

sys.path.append('/Users/lucfaessler/Documents/Studium/Bachelor/Wirtschaftsinformatik/Semester 8 - SS25/Practical Course NLP/nlp_practical_2025_sEXism/code')
sys.path.append('/Users/lucfaessler/Documents/Studium/Bachelor/Wirtschaftsinformatik/Semester 8 - SS25/Practical Course NLP/nlp_practical_2025_sEXism/data')
from data_loader.single_task_dataset import SingleTaskDataset
from data_loader.vocab import build_vocab
from train.torch_train import train_model
from preprocessing.utils import preprocess_dataframe
from models.new.pytorch.vanilla_models import textcnn, bilstm, bigru
from models.new.pytorch.attention_models import *


df1 = pd.read_csv("../../../../data/en/encoded/aeda_encoded.csv")
df2 = pd.read_csv("../../../../data/en/encoded/backtranslated_encoded.csv")
df3 = pd.read_csv("../../../../data/en/encoded/translated_encoded.csv")

df_train = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

df_train = preprocess_dataframe(df_train, 'tweet')

label_column = 'gold_labels_task1_2'

df_train = df_train[df_train[label_column].notna()]

# 1. Preprocess text column

# 2. Split train into train+val
train_df, val_df = train_test_split(df_train, test_size=0.1, stratify=df_train[label_column])

train_df[:5]

,Unnamed: 0,id_EXIST,lang,tweet,number_annotators,annotators,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,labels_task1_1,labels_task1_2,labels_task1_3,split,source,gold_labels_task1_1,gold_labels_task1_2,gold_labels_task1_3,translated_text
5130,1870,201871,en,23 citizens have just died and why do employee...,6,"['Annotator_187', 'Annotator_635', 'Annotator_...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['White or Caucasian', 'White or Caucasian', '...","['Bachelor’s degree', 'Bachelor’s degree', 'Ba...","['Spain', 'Argentina', 'Mexico', 'Poland', 'Ca...","['NO', 'NO', 'NO', 'NO', 'NO', 'NO']","['-', '-', '-', '-', '-', '-']","[['-'], ['-'], ['-'], ['-'], ['-'], ['-']]",TRAIN_EN,NaN,0.0,0.0,"[1, 0, 0, 0, 0, 0]",NaN
3204,3204,203205,en,"to the guy who beeped ; at me , , after id put...",6,"['Annotator_645', 'Annotator_646', 'Annotator_...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['White or Caucasian', 'White or Caucasian', '...","['Bachelor’s degree', 'Bachelor’s degree', 'Ba...","['Portugal', 'United States', 'Portugal', 'Sou...","['YES', 'NO', 'NO', 'NO', 'YES', 'NO']","['REPORTED', '-', '-', '-', 'DIRECT', '-']","[['STEREOTYPING-DOMINANCE'], ['-'], ['-'], ['-...",TRAIN_EN,aeda,0.0,0.0,"[1, 0, 0, 0, 0, 0]",NaN
560,560,200561,en,i . think about this every . time ? the ! resp...,6,"['Annotator_43', 'Annotator_596', 'Annotator_5...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['Hispano or Latino', 'Black or African Americ...","['High school degree or equivalent', 'Master’s...","['Mexico', 'South Africa', 'Portugal', 'Estoni...","['NO', 'NO', 'NO', 'NO', 'NO', 'NO']","['-', '-', '-', '-', '-', '-']","[['-'], ['-'], ['-'], ['-'], ['-'], ['-']]",TRAIN_EN,aeda,0.0,0.0,"[1, 0, 0, 0, 0, 0]",NaN
3253,3253,203254,en,"maam if , i . say that you look like . a whore...",6,"['Annotator_668', 'Annotator_669', 'Annotator_...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['Hispano or Latino', 'other', 'White or Cauca...","['High school degree or equivalent', 'Master’s...","['Mexico', 'Algeria', 'Portugal', 'Spain', 'Un...","['YES', 'YES', 'NO', 'YES', 'YES', 'YES']","['DIRECT', 'DIRECT', '-', 'REPORTED', 'DIRECT'...","[['OBJECTIFICATION', 'MISOGYNY-NON-SEXUAL-VIOL...",TRAIN_EN,aeda,1.0,1.0,"[0, 0, 0, 1, 1, 0]",NaN
462,462,200463,en,because your the boss! strong beautiful and po...,6,"['Annotator_496', 'Annotator_497', 'Annotator_...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['Hispano or Latino', 'White or Caucasian', 'W...","['High school degree or equivalent', 'Bachelor...","['Portugal', 'Portugal', 'Poland', 'Mexico', '...","['NO', 'NO', 'NO', 'NO', 'NO', 'NO']","['-', '-', '-', '-', '-', '-']","[['-'], ['-'], ['-'], ['-'], ['-'], ['-']]",TRAIN_EN,aeda,0.0,0.0,"[1, 0, 0, 0, 0, 0]",NaN


In [69]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from collections import Counter
import re
import numpy as np
import gensim.downloader as api
from sklearn.metrics import f1_score

# --- Preprocessing function ---
def preprocess_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"@\w+", '', text)
    text = re.sub(r"#\w+", '', text)
    return text

# --- Dataset Class ---
class SentimentDataset(Dataset):
    def __init__(self, df, vocab, max_length, label_columns=['YES', 'NO']):
        #self.texts = df['text'].values
        #self.labels = df['YES'].values.astype(bool)
        self.texts = df['tweet'].values
        #self.labels = df['gold_labels_task1_1'].values.astype(bool)
        self.labels = df[label_column].astype(int).values
        self.vocab = vocab
        self.max_length = max_length
        self.pad_token_id = self.vocab.get('<PAD>', 0)
        print(self.labels[:5])

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        tokens = self.tokenize_text(text)
        label = self.labels[idx]
        return {
            #'input_ids': torch.tensor(tokens, dtype=torch.long),
            #'labels': torch.tensor(self.labels[idx], dtype=torch.float)
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            #'labels': torch.tensor(int(self.labels[idx]), dtype=torch.long)  # class index: 0 or 1
            "labels": torch.tensor(label, dtype=torch.long)

        }

    def tokenize_text(self, text):
        words = re.findall(r'\b\w+\b', text.lower())
        tokens = [self.vocab.get(word, self.vocab.get('<UNK>', 1)) for word in words]

        if len(tokens) < self.max_length:
            tokens.extend([self.pad_token_id] * (self.max_length - len(tokens)))
        else:
            tokens = tokens[:self.max_length]
        return tokens

# --- Vocabulary building with gensim glove-twitter ---
def build_vocab(texts, word2vec_model=None, min_freq=2, embedding_dim=200):
    all_words = []
    for text in texts:
        words = re.findall(r'\b\w+\b', str(text).lower())
        all_words.extend(words)

    word_counts = Counter(all_words)

    vocab = {'<PAD>': 0, '<UNK>': 1}
    embeddings = [np.zeros(embedding_dim, dtype=np.float32),  # PAD embedding
                  np.random.normal(scale=0.6, size=embedding_dim).astype(np.float32)]  # UNK embedding

    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
            if word2vec_model and word in word2vec_model.key_to_index:
                embeddings.append(word2vec_model[word].astype(np.float32))
            else:
                embeddings.append(np.random.normal(scale=0.6, size=embedding_dim).astype(np.float32))

    embedding_matrix = torch.tensor(np.array(embeddings), dtype=torch.float32)
    return vocab, embedding_matrix

# --- Training function ---
def train_model(model, train_df, eval_df, device,
                max_length=100, epochs=5, batch_size=32, lr=0.001,
                weight_decay=1e-5, clip_grad_norm=1.0):

    vocab = build_vocab(train_df['tweet'])[0]
    pad_token_id = vocab.get('<PAD>', 0)

    train_dataset = SentimentDataset(train_df, vocab, max_length, label_columns=['YES'])
    eval_dataset = SentimentDataset(eval_df, vocab, max_length, label_columns=['YES'])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

    #criterion = nn.BCEWithLogitsLoss()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    model.to(device)
    print(f"Training on {device}...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            #labels = batch['labels'].unsqueeze(1).to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids)
            loss = criterion(logits, labels)
            loss.backward()

            if clip_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)

            optimizer.step()
            train_loss += loss.item()

        model.eval()
        eval_loss = 0
        correct = 0
        total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in eval_loader:
                """
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].unsqueeze(1).to(device)

                logits = model(input_ids)
                loss = criterion(logits, labels)
                eval_loss += loss.item()

                probs = torch.sigmoid(logits)
                predicted = (probs > 0.5).float()

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                """

                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)  # already shape [B]

                logits = model(input_ids)  # shape: [B, 2]
                loss = criterion(logits, labels)
                eval_loss += loss.item()

                predicted = torch.argmax(logits, dim=1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())


        # Compute F1 score
        #f1 = f1_score(all_labels, all_preds, average='binary')  # Binary F1
        macro_f1 = f1_score(all_labels, all_preds, average='macro')  # Macro F1
        unique_classes = np.unique(all_labels)
        per_class_f1 = f1_score(all_labels, all_preds, average=None, labels=unique_classes)
        micro_f1 = f1_score(all_labels, all_preds, average='micro')
        weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

        print(f'Epoch {epoch+1}/{epochs}:')
        print(f'  Train Loss: {train_loss/len(train_loader):.4f}')
        print(f'  Eval Loss: {eval_loss/len(eval_loader):.4f}')
        print(f'  Eval Accuracy: {100*correct/total:.2f}%')
        #print(f'  Eval F1 Score (binary): {f1:.4f}')
        print(f'  Eval F1 Score (macro):  {macro_f1:.4f}')
        print(f'  Eval F1 Score (macro):  {micro_f1:.4f}')
        print(f'  Eval F1 Score (macro):  {weighted_f1:.4f}')
        #print(f'  Per-Class F1 Scores: class 0 = {per_class_f1[0]:.4f}, class 1 = {per_class_f1[1]:.4f}')
        print(f'  Per-Class F1 Scores: class 0 = {per_class_f1[0]:.4f}, class 1 = {per_class_f1[1]:.4f}, class 2 = {per_class_f1[2]:.4f}, class 3 = {per_class_f1[3]:.4f}')
        print("-" * 30)


In [70]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes, 
                 filter_sizes=(3, 4, 5), num_filters=100, dropout_prob=0.5, 
                 pretrained_embeddings=None, freeze_embeddings=False):
        super().__init__()

        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=freeze_embeddings)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, 
                      out_channels=num_filters, 
                      kernel_size=fs)
            for fs in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes - 1)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = embedded.permute(0, 2, 1) 
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]
        cat = self.dropout(torch.cat(pooled, dim=1))
        output = self.fc(cat)
        return output

In [71]:
import nltk
from nltk.tokenize import word_tokenize
import gensim.downloader as api
import os
sys.path.append(os.path.abspath('../'))

from pytorch.attention_models import *
from pytorch.vanilla_models import bigru, textcnn, bilstm

def preprocess_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"@\w+", '', text)
    text = re.sub(r"#\w+", '', text)
    #text = re.sub(r"[^a-z\s]", '', text)
    #tokens = nltk.word_tokenize(text)
    return text


df_t = train_df.copy()
df_e = val_df.copy()

df_t['tweet'] = df_t['tweet'].apply(preprocess_tweet)
df_e['tweet'] = df_e['tweet'].apply(preprocess_tweet)

# Build vocab and embedding matrix using pretrained vectors
vocab, embedding_matrix = build_vocab(df_t['tweet'], word2vec_model=word2vec, embedding_dim=100)

# Instantiate model with pretrained embeddings, freezing embeddings
#model = textcnn.BiGRUClassifier(vocab_size=len(vocab),
#                embedding_dim=embedding_matrix.shape[1],
#                hidden_dim=128,
#                num_classes=2,
#                num_layers=2,
#                pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
#                freeze_embeddings=False)

model = textcnn.TextCNN(vocab_size=len(vocab),
                embedding_dim=embedding_matrix.shape[1],
                num_classes=4,
                pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
                freeze_embeddings=False)

# Train model
train_model(model, df_t, df_e, device=torch.device('cpu'), epochs=15)

/var/folders/0s/37b847w521n_1v4k0x2wrrlr0000gn/T/ipykernel_20131/3248967889.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),


[0 0 0 1 0]
[0 0 3 0 0]
Training on cpu...
Epoch 1/15:
  Train Loss: 0.8620
  Eval Loss: 0.7851
  Eval Accuracy: 69.83%
  Eval F1 Score (macro):  0.3597
  Eval F1 Score (macro):  0.6983
  Eval F1 Score (macro):  0.6457
  Per-Class F1 Scores: class 0 = 0.8361, class 1 = 0.4932, class 2 = 0.1096, class 3 = 0.0000
------------------------------
Epoch 2/15:
  Train Loss: 0.6544
  Eval Loss: 0.7369
  Eval Accuracy: 72.29%
  Eval F1 Score (macro):  0.5223
  Eval F1 Score (macro):  0.7229
  Eval F1 Score (macro):  0.7193
  Per-Class F1 Scores: class 0 = 0.8454, class 1 = 0.6311, class 2 = 0.3579, class 3 = 0.2549
------------------------------
Epoch 3/15:
  Train Loss: 0.5123
  Eval Loss: 0.6764
  Eval Accuracy: 74.01%
  Eval F1 Score (macro):  0.4901
  Eval F1 Score (macro):  0.7401
  Eval F1 Score (macro):  0.7203
  Per-Class F1 Scores: class 0 = 0.8648, class 1 = 0.6265, class 2 = 0.3111, class 3 = 0.1579
------------------------------
Epoch 4/15:
  Train Loss: 0.3790
  Eval Loss: 0.6276
 

In [ ]:
"""
CNN
Epoch 5/10:
  Train Loss: 0.1034
  Eval Loss: 0.3146
  Eval Accuracy: 88.70%
  Eval F1 Score (binary): 0.8713
  Eval F1 Score (macro):  0.8853
  Per-Class F1 Scores: class 0 = 0.8993, class 1 = 0.8713
"""
"""
LSTM-1
Epoch 8/10:
  Train Loss: 0.0446
  Eval Loss: 0.5588
  Eval Accuracy: 85.12%
  Eval F1 Score (binary): 0.8248
  Eval F1 Score (macro):  0.8478
  Per-Class F1 Scores: class 0 = 0.8707, class 1 = 0.8248
"""
"""
LSTM-2
Epoch 7/10:
  Train Loss: 0.0670
  Eval Loss: 0.6380
  Eval Accuracy: 86.02%
  Eval F1 Score (binary): 0.8295
  Eval F1 Score (macro):  0.8555
  Per-Class F1 Scores: class 0 = 0.8815, class 1 = 0.8295
"""
"""
GRU-1
Epoch 9/10:
  Train Loss: 0.0265
  Eval Loss: 0.6529
  Eval Accuracy: 86.35%
  Eval F1 Score (binary): 0.8360
  Eval F1 Score (macro):  0.8596
  Per-Class F1 Scores: class 0 = 0.8831, class 1 = 0.8360
"""
"""
GRU-2
Epoch 7/10:
  Train Loss: 0.0466
  Eval Loss: 0.6475
  Eval Accuracy: 85.68%
  Eval F1 Score (binary): 0.8298
  Eval F1 Score (macro):  0.8531
  Per-Class F1 Scores: class 0 = 0.8764, class 1 = 0.8298
"""


#embedding = load_embeddings("glove-twitter-200")
embedding = api.load("glove-twitter-100")

# 3. Build vocab and embedding matrix
vocab, embedding_matrix = build_vocab(train_df["tweet"], embedding=embedding, embedding_dim=200)

# 4. Create Dataset and DataLoader
train_ds = SingleTaskDataset(train_df, vocab, max_length=100, label_column=label_column, task='binary')
val_ds   = SingleTaskDataset(val_df,   vocab, max_length=100, label_column=label_column, task='binary')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)


#model = textcnn.TextCNN(
#    vocab_size=len(vocab),
#    embedding_dim=embedding_matrix.shape[1],
#    num_classes=1,
#    pretrained_embeddings=embedding_matrix
#)

model = textcnn.TextCNN(vocab_size=len(vocab),
                embedding_dim=embedding_matrix.shape[1],
                num_classes=2,
                pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
                freeze_embeddings=False)

# 6. Loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.9)

# 7. Train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    loss_fn,
    device,
    task_type='binary',
    epochs=5,
    use_wandb=False,  # change to True to enable Weights & Biases
    run_name="textcnn_multiclass"
)
